In [ ]:
# %% [markdown]
# # Exploratory Data Analysis (EDA)
# This notebook is for inspecting the dataset, checking class balance, and visualizing feature correlations before running the main training loop.

# %%
import sys
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Add the 'src' directory to the system path so we can import our modules
sys.path.append(os.path.abspath('../src'))

# Import your custom modules
import dataset
import vae

# Set style
sns.set(style="whitegrid")

# %% [markdown]
# ## 1. Inspect Audio Data (Easy Task)
# Let's load the raw CSV files directly to look at the column names and distributions.

# %%
# Define paths (Adjust if your folder structure is different)
path_bangla = "../data/audio/dataset.csv"
path_english = "../data/audio/features_30_sec.csv"

if os.path.exists(path_bangla) and os.path.exists(path_english):
    df_b = pd.read_csv(path_bangla)
    df_e = pd.read_csv(path_english)
    
    print(f"Bangla Data Shape: {df_b.shape}")
    print(f"English Data Shape: {df_e.shape}")
    
    # Show first few rows of Bangla data
    display(df_b.head(3))
else:
    print("Data files not found. Check your paths.")

# %% [markdown]
# ### Class Balance Check
# We need to ensure we have roughly equal amounts of data for both languages.

# %%
# Simple count plot
count_data = pd.DataFrame({
    'Language': ['Bangla', 'English'],
    'Count': [len(df_b), len(df_e)]
})

plt.figure(figsize=(6, 4))
sns.barplot(data=count_data, x='Language', y='Count', palette='viridis')
plt.title("Total Available Samples per Language")
plt.show()

# %% [markdown]
# ## 2. Visualize Feature Correlations
# Strong correlations between features (like `mfcc1` and `mfcc2`) might help the VAE learn better.

# %%
# Select just the MFCC columns for a cleaner heatmap
mfcc_cols = [col for col in df_b.columns if 'mfcc' in col][:10] # First 10 MFCCs
subset = df_b[mfcc_cols].dropna()

plt.figure(figsize=(10, 8))
sns.heatmap(subset.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Matrix of MFCC Features (Bangla)")
plt.show()

# %% [markdown]
# ## 3. Test Data Loading Pipeline
# Now we test if `dataset.py` works correctly by calling its functions.

# %%
# Test loading the Easy Data
try:
    X_scaled, y_true = dataset.load_easy_data()
    print("\n[Success] Easy Data Loaded via src/dataset.py")
    print(f"Input Shape: {X_scaled.shape}")
    print(f"Labels Shape: {y_true.shape}")
    print(f"Class distribution in loaded batch: {np.bincount(y_true)} (0=Eng, 1=Ban)")
except Exception as e:
    print(f"[Error] Could not load easy data: {e}")

# %% [markdown]
# ## 4. Inspect Text Embeddings (Medium/Hard Task)
# Check if the BERT embeddings have been generated and what they look like.

# %%
import torch

emb_path = "bangla_embeddings.pt" # Or "../data/lyrics/bangla_embeddings.pt"

if os.path.exists(emb_path):
    tensor = torch.load(emb_path)
    print(f"Bangla Embeddings Loaded: {tensor.shape}")
    
    # Convert to numpy for plotting
    emb_np = tensor.numpy()
    
    # Plot variance of the first 50 embedding dimensions
    plt.figure(figsize=(12, 4))
    plt.plot(np.var(emb_np, axis=0)[:100])
    plt.title("Variance of BERT Embedding Dimensions (First 100)")
    plt.xlabel("Dimension Index")
    plt.ylabel("Variance")
    plt.show()
else:
    print(f"Embedding file {emb_path} not found. Run main.py first to generate them.")

# %% [markdown]
# ## 5. Model Architecture Check
# Visualize the complex CVAE architecture to ensure the layers are connected correctly.

# %%
# Build the model using your src/vae.py script
encoder, decoder = vae.build_cvae()

print("=== Encoder Summary ===")
encoder.summary()

print("\n=== Decoder Summary ===")
decoder.summary()

# %%